# 🚀 ORLITH AI — Google Colab L4 GPU Server Setup

Notebook ini mempersiapkan dan menjalankan **Backend Orlith (DocuMind AI)** secara penuh di Google Colab menggunakan akselerasi **NVIDIA L4 GPU (24GB VRAM)**.

### Komponen yang berjalan di dalam Colab:
1. **Local LLM Engine**: **Ollama** (Qwen 2.5 7B / 14B atau Llama 3.1 8B) berjalan langsung di VRAM L4.
2. **Local Embedding & Reranker**: `BAAI/bge-m3` dan `BAAI/bge-reranker-base` berjalan dengan PyTorch CUDA.
3. **OCR Engine**: EasyOCR (Bahasa Indonesia & Inggris) berjalan di CUDA.
4. **FastAPI Backend Server**: Menjalankan seluruh pipeline RAG, Hybrid Search, Chunks, dan API.
5. **Storage**: Menyimpan database (`documind.db`), ChromaDB, dan dokumen di disk lokal Colab `/content/orlith_data`.
6. **Cloudflare Tunnel (`cloudflared`)**: Menghasilkan URL publik HTTPS gratis tanpa akun untuk dihubungkan ke Frontend Anda (Vercel atau localhost).

## ⚙️ Langkah 1: Cek Akselerator GPU
Pastikan runtime Anda sudah menggunakan **GPU L4** (*Runtime -> Change runtime type -> Hardware accelerator: GPU -> GPU type: L4*).

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 💾 Langkah 2: Setup Storage Data Lokal Colab
Menyiapkan folder penyimpanan lokal di `/content/orlith_data` untuk database SQLite, ChromaDB vector store, dan file uploads.

In [ ]:
import os

PERSIST_DATA_DIR = "/content/orlith_data"
os.makedirs(f"{PERSIST_DATA_DIR}/chroma", exist_ok=True)
os.makedirs(f"{PERSIST_DATA_DIR}/uploads", exist_ok=True)
print(f"✅ Direktori data siap digunakan: {PERSIST_DATA_DIR}")

## 📥 Langkah 3: Ekstrak Source Code Backend (`backend.zip`)
Silakan drag-and-drop file **`backend.zip`** dari laptop Anda ke panel **Files (📁)** di sidebar sebelah kiri Colab, lalu jalankan cell ini.

In [ ]:
import os
import sys

ZIP_PATH = "/content/backend.zip"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        "❌ File /content/backend.zip TIDAK DITEMUKAN!\n"
        "👉 Silakan drag-and-drop file 'backend.zip' dari laptop Anda ke panel Files (📁) di sebelah kiri Colab, lalu jalankan ulang cell ini."
    )

print("📦 Mengekstrak backend.zip...")
!rm -rf /content/orlith_backend /content/backend
!unzip -q -o /content/backend.zip -d /content
if os.path.exists("/content/backend"):
    !mv /content/backend /content/orlith_backend

print("✅ Backend berhasil diekstrak ke /content/orlith_backend!")

# Masuk ke direktori backend
%cd /content/orlith_backend

## 📦 Langkah 4: Install Dependencies & Cloudflare Tunnel
Install dependensi sistem (`zstd`, `curl`, `pciutils`), `requirements.txt` backend, PyTorch CUDA 12.1, dan binary `cloudflared`.

In [ ]:
%%bash
echo "=== Menginstall dependensi sistem (zstd, curl, pciutils) ==="
apt-get update -qq && apt-get install -y -qq zstd curl pciutils

echo "=== Menginstall Python dependencies ==="
pip install --quiet -r requirements.txt
pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install --quiet sentence-transformers

echo "=== Mengunduh Cloudflare Tunnel (cloudflared) ==="
if ! command -v cloudflared &> /dev/null; then
    wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    rm -f cloudflared-linux-amd64.deb
fi
echo "✅ Semua dependensi & cloudflared berhasil disiapkan!"

## 🦙 Langkah 5: Install & Jalankan Ollama Engine (GPU Background Daemon)
Ollama akan berjalan di latar belakang memanfaatkan GPU NVIDIA L4 pada port `11434`.

In [ ]:
%%bash
# Install dependensi sistem yang dibutuhkan Ollama
apt-get update -qq && apt-get install -y -qq zstd curl pciutils

# Install Ollama CLI
if ! command -v ollama &> /dev/null; then
    curl -fsSL https://ollama.com/install.sh | sh
fi

# Jalankan daemon Ollama di background
pkill ollama || true
nohup ollama serve > /content/ollama.log 2>&1 &
echo "✅ Daemon Ollama sedang dijalankan di background..."

### Pull Model Lokal ke Ollama
Rekomendasi untuk GPU L4 (24GB VRAM):
- `qwen2.5:7b` (Sangat cepat ~4.5GB VRAM, terbaik untuk Bahasa Indonesia & JSON)
- `qwen2.5:14b` (Akurasi RAG tinggi, butuh ~9GB VRAM — L4 sangat sanggup)
- `llama3.1:8b`

In [ ]:
import time, urllib.request

# Tunggu hingga Ollama merespons
for _ in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434/")
        print("✅ Ollama daemon aktif dan siap digunakan!")
        break
    except Exception:
        time.sleep(1)

# Model yang akan diunduh ke GPU
CHOSEN_MODEL = "qwen2.5:7b"
print(f"Mengunduh model {CHOSEN_MODEL}...")
!ollama pull {CHOSEN_MODEL}

!ollama list

## ⚙️ Langkah 6: Konfigurasi Environment File (`.env`)
Mengatur backend untuk menggunakan:
- LLM Lokal: `qwen2.5:7b` via Ollama
- Embedding Lokal: `BAAI/bge-m3` via HuggingFace Sentence-Transformers (CUDA)
- Reranker Lokal: `BAAI/bge-reranker-base` (CUDA)
- Storage lokal di `/content/orlith_data`

In [ ]:
import secrets

secret_key = secrets.token_hex(32)
env_content = f"""ENVIRONMENT=development
SECRET_KEY={secret_key}
DATABASE_URL=sqlite+aiosqlite:///{PERSIST_DATA_DIR}/documind.db
CHROMA_PERSIST_DIR={PERSIST_DATA_DIR}/chroma
STORAGE_BACKEND=local
STORAGE_LOCAL_PATH={PERSIST_DATA_DIR}/uploads

# AI Model Configuration (100% Local GPU)
LLM_PROVIDER=ollama
LLM_MODEL=qwen2.5:7b
OLLAMA_BASE_URL=http://localhost:11434
EMBEDDING_PROVIDER=huggingface
EMBEDDING_MODEL=BAAI/bge-m3

# RAG Pipeline Settings
ENABLE_RERANKER=true
RERANKER_MODEL=BAAI/bge-reranker-base
ENABLE_SEMANTIC_CHUNKING=true
ENABLE_HYBRID_SEARCH=true
ENABLE_PARENT_CHILD_CHUNKING=true

# Allow Frontend Connections from anywhere
CORS_ORIGINS=*
LOG_LEVEL=INFO
"""

with open("/content/orlith_backend/.env", "w") as f:
    f.write(env_content)

print("✅ Konfigurasi .env berhasil disimpan!")

## 🌐 Langkah 7: Jalankan Cloudflare Tunnel & FastAPI Backend Server
Cell ini akan:
1. Membuka tunnel Cloudflare ke port `8000`.
2. Menampilkan URL publik HTTPS yang bisa Anda gunakan di Frontend.
3. Menjalankan server backend Uvicorn secara aktif.

In [ ]:
import subprocess
import time
import re

# 1. Jalankan Cloudflare Tunnel di background
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# 2. Tangkap Public HTTPS URL dari log cloudflared
public_url = None
time.sleep(3)
for _ in range(20):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("=" * 65)
if public_url:
    print(f"🎉 PUBLIC BACKEND URL ANDA: {public_url}")
    print(f"📑 Interactive API Docs   : {public_url}/docs")
    print(f"❤️ Health Check Endpoint  : {public_url}/health")
    print("=" * 65)
    print("\n👉 Masukkan URL ini ke file .env Frontend Anda (localhost / Vercel):")
    print(f"NEXT_PUBLIC_API_URL={public_url}\n")
else:
    print("Tunnel sedang diinisialisasi, cek URL di log tunnel...")
print("=" * 65)

# 3. Jalankan FastAPI Server
!python -m uvicorn main:app --host 0.0.0.0 --port 8000